### Inicializando las librerias a ocupar ###

In [5]:
import pyarrow.parquet as pq
import gcsfs
import pandas as pd
from pandas_gbq import to_gbq
import bigframes.pandas as bpd
import json
from google.cloud import storage
from google.cloud import bigquery


In [9]:
def get_table_curated_by_columns(table):
    # Initialize BigQuery client
    client = bigquery.Client.from_service_account_json("acme-987654-c052039ac4cd.json")

    # Define your query
    query = f"""
        SELECT latitude, longitude FROM acme-987654.Curated.{table} limit 4
    """

    # Run the query
    print(query)
    query_job = client.query(query)
    df = query_job.to_dataframe()
    return df

In [10]:
def get_coordinates():
    # Create a map centered on Mexico City
    df = get_table_curated_by_columns("Estados")
    

    #map_mx = folium.Map(location=[29.954423,-90.0714173], zoom_start=12)

    # Define the polygon coordinates
    #polygon_coords = [df.to_numpy()]

    # Add a semi-transparent polygon to create a shadow effect
    #folium.Polygon(
    #    locations=polygon_coords,
    #    color="black",
    #    fill=True,
    #    fill_color="gray",
    #    fill_opacity=0.3  # Shadow transparency
    #).add_to(map_mx)

    # Display the map inside Jupyter Notebook
    #return map_mx
    #return ""
    hashable_key = tuple(df.loc[0:4].flatten()) 
    return hashable_key

In [11]:
get_coordinates()

FileNotFoundError: [Errno 2] No such file or directory: 'acme-987654-c052039ac4cd.json'

### Librerias a instalar ###

In [31]:
#pip install bigframes
#pip install pandas-gbq
#pip install google-cloud-bigquery
#pip install  --upgrade bigframes

### Información BIG QUERY

In [ ]:
PROJECT_ID = 'adsac-455509'
LOCATION = "southamerica-east1" 

### insertando opciones de BigQuery DataFrames

In [33]:
bpd.options.bigquery.project = PROJECT_ID
bpd.options.bigquery.location = LOCATION
bpd.options.display.progress_bar = None

## Starting with Business picklet ##

In [34]:
# Create a GCS file system object
fs = gcsfs.GCSFileSystem(project=PROJECT_ID)

# Path to your Parquet file in GCS
file_path = 'gs://adsac/Yelp/business.pkl'
df_business = pd.read_pickle(file_path)

In [35]:
#Eliminando duplicados
df_business = df_business.loc[:, ~df_business.columns.duplicated()]

#Actualizando tipos
df_business['attributes'] = df_business['attributes'].astype(str)
df_business['hours'] = df_business['hours'].astype(str)

In [36]:
#Eliminando nulos
df_business = df_business[df_business['business_id'].isnull() == False]
df_business = df_business[df_business['latitude'].isnull() == False]
df_business = df_business[df_business['longitude'].isnull() == False]
df_business = df_business[df_business['stars'].isnull() == False]
df_business = df_business[df_business['review_count'].isnull() == False]
df_business = df_business[df_business['is_open'].isnull() == False]


In [37]:
#Insertar en la tabla curada
destination_table = 'adsac-455509.Curated.Business'

# Upload DataFrame
to_gbq(df_business, destination_table, project_id=PROJECT_ID, if_exists='replace')  # Use 'append' if you want to add rows

## Para leer la tabla curada en big query

In [38]:
# This is how you read a BigQuery table
#df_business_curated = bpd.read_gbq("adsac-455509.Curated.Business")
#df_business_curated.peek()

### Leyendo el archivo tipo Parquet ###

In [39]:
# Create a GCS file system object
fs = gcsfs.GCSFileSystem(project=PROJECT_ID)

# Path to your Parquet file in GCS
file_path = 'gs://adsac/Yelp/user.parquet'

# Open the file and read it
with fs.open(file_path) as f:
    parquet_file = pq.ParquetFile(f)
    df_user = parquet_file.read().to_pandas()

print(df_user.head())

KeyboardInterrupt: 

## Borrando columnas y filas inecesarias 

In [ ]:
#Eliminando archivos 
df_user = df_user[df_user['review_count'] > 0]
df_user = df_user.drop(columns=['compliment_more', 'compliment_profile', 'compliment_cute', 'compliment_list', 'compliment_note', 'compliment_plain', 'compliment_cool'])
df_user = df_user.drop(columns=['compliment_hot', 'compliment_funny', 'compliment_writer', 'compliment_photos'])


### Insertando en BigQuery

In [ ]:
#Insertar en la tabla curada
destination_table = 'adsac-455509.Curated.Usuario'

# Upload DataFrame
to_gbq(df_user, destination_table, project_id=PROJECT_ID, if_exists='replace')  # Use 'append' if you want to add rows

### Para leer la tabla curada en big query

In [ ]:
# This is how you read a BigQuery table
#df_user_curated = bpd.read_gbq("adsac-455509.Curated.Usuario")
#df_user_curated.peek()

,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,average_stars
798403,cX6CwuUFV_B5maGj6zlXkw,Sei,1,2018-09-18 22:39:19,0,0,0,,None,0,1.0
798534,9iJcS_yAf4kDGwyrRGi_5A,Teddy,1,2014-06-22 00:52:31,0,0,0,,None,0,5.0
798565,AAxw_r1S3SBlydZfM0AOoQ,Liz,3,2014-10-08 15:04:10,2,0,0,,None,0,5.0
798263,RVcoVbbrQP9me1brqHeJ2Q,Kenneth,4,2012-06-12 17:37:37,3,1,2,,"OPc_47Iuzb8doUQbhiKuSA, lUr1OnGz8n1Q3S0l9tdMUQ...",0,3.75
798594,Pxvu4OU8vuyVJAIkTA37gg,Thuong,217,2017-01-01 00:02:56,166,41,90,"2019,20,20,2021","bwF0z1y_srWijUwmi7_kKQ, 6_i6eYOznkIZty18j5vuZg...",9,3.47


### CHECK IN JSON

In [51]:
# This is how you read a BigQuery table
df_checkin = bpd.read_gbq("adsac-455509.Staging.CheckIn")
df_checkin_pandas = df_checkin.to_pandas()
df_checkin_pandas = df_checkin_pandas.drop_duplicates()

In [ ]:
#Eliminando los nulls
df_checkin_pandas = df_checkin_pandas[df_checkin_pandas['business_id'].isnull() == False]
df_checkin_pandas = df_checkin_pandas[df_checkin_pandas['date'].isnull() == False]

#### Leer tabla curada

In [67]:
#This is how you read a BigQuery table
df_checkin_pandas = bpd.read_gbq("adsac-455509.Curated.CheckIn")
df_checkin_pandas.peek()

,date,business_id
95009,"2010-11-27 04:05:50, 2010-12-20 04:34:26, 2011...",d7W1Fi6uRYUhDT_uW83d7w
94669,2011-04-30 15:43:55,SezTh5uY5IY1OZEracWmMw
94990,"2019-11-16 19:48:55, 2020-04-07 16:50:19, 2020...",a6q_KuJs285A4m3YnBVLjQ
95018,"2016-03-02 01:00:43, 2016-03-02 20:48:34, 2016...",SOsjW1JARmtHUFtpFlp8rw
94567,"2014-09-17 19:31:02, 2014-09-22 20:45:14, 2014...",pHmGdzi7B2NpkWR1YKtVbg


#### Agregando la tabla curada a Google Storage

In [ ]:
destination_table = 'adsac-455509.Curated.CheckIn'

# Upload DataFrame
to_gbq(df_checkin_pandas, destination_table, project_id=PROJECT_ID, if_exists='replace')  # Use 'append' if you want to add rows

### Reviews

In [54]:
# This is how you read a BigQuery table
df_review = bpd.read_gbq("adsac-455509.Staging.Review")
df_review_pandas = df_review.to_pandas()
df_review_pandas = df_review_pandas.drop_duplicates()

In [60]:
df_review_pandas = df_review_pandas[df_review_pandas['business_id'].isnull() == False]
df_review_pandas = df_review_pandas[df_review_pandas['user_id'].isnull() == False]
df_review_pandas = df_review_pandas[df_review_pandas['text'].isnull() == False]

In [64]:
df_review_pandas = df_review_pandas.drop(columns=['cool', 'useful','funny'])

In [66]:
destination_table = 'adsac-455509.Curated.Review'

# Upload DataFrame
to_gbq(df_review_pandas, destination_table, project_id=PROJECT_ID, if_exists='replace')  # Use 'append' if you want to add rows

### Obtener tabla Curada

In [68]:
#This is how you read a BigQuery table
df_checkin_pandas = bpd.read_gbq("adsac-455509.Curated.Review")
df_checkin_pandas.peek()

,text,stars,date,review_id,business_id,user_id
1726811,"First off, I loved the atmosphere! We walked i...",3.0,2011-10-09 18:33:37+00:00,mN55rTSsmWPpNgW-4OsF1Q,CxzaEZX7Zu8xHBqOBlzqFQ,ieddiannWKFvVmBGS778Vw
1726343,I always come stop by for donuts and sticky bu...,3.0,2015-11-27 02:41:31+00:00,BKPdccRAr8zeDfJmDQubeA,KCVv4CFsiWZnIMaLdGteuQ,UqyP8F6MRg5p2ZAr5CXz1Q
1726598,"Ate lunch here the other day, while in town fo...",4.0,2014-07-24 12:55:49+00:00,HiWj6Y0247zfIK25mjzEwA,WrgdQF8kzvONbZctSPlF4A,Snk_n5DNp_mHuWDVH_nMmg
1726163,Best West Indian (Southern Caribbean) restaura...,5.0,2017-11-18 16:33:38+00:00,5AHrAaUrgtt3wuRHU1of9A,ZuwJdk2g6XGRWV94TE50Cg,9lToW9IE_KmWNE8aOzXRzA
1726241,Took my truck in for an oil leak . Had to retu...,5.0,2018-02-19 19:02:27+00:00,ogHcaYdxmsaSdiFp-euP3A,oAIGgWkdVz9CL5J_2RhznQ,EBC3iaiDs10Lcc0SsU3p0w


### Sitios

In [74]:
# Initialize the Google Cloud Storage client
client = storage.Client()

# Specify your bucket and file details
bucket_name = 'gs://adsac/Google'
file_name = '1.json'

# Access the bucket and blob (file)
bucket = client.bucket(bucket_name)
blob = bucket.blob(file_name)

# Download the JSON file as a string
json_data = blob.download_as_text()

# Parse the JSON string into a Python dictionary
data = json.loads(json_data)

# Print the data
print(data)

NotFound: 404 GET https://storage.googleapis.com/download/storage/v1/b/gs://adsac/Google/o/1.json?alt=media: Not Found: ('Request failed with status code', 404, 'Expected one of', <HTTPStatus.OK: 200>, <HTTPStatus.PARTIAL_CONTENT: 206>)

In [2]:
from functionBigQuery import etl_review_json_file
df_review = etl_review_json_file('Yelp/review.json') 

Project is:  acme-987654

        MERGE `acme-987654.Curated.Review` AS target
        USING `acme-987654.Raw.Review` AS source
        ON target.business_id = source.business_id 
            and target.user_id = source.user_id
            and target.review_id = source.review_id
        WHEN MATCHED THEN
            UPDATE SET target.date = source.date
        WHEN NOT MATCHED THEN
            INSERT (review_id, user_id, business_id, stars, useful, funny, cool, text, date) 
            VALUES (source.review_id, source.user_id, source.business_id, source.stars, source.useful, source.funny, source.cool, source.text, source.date)
        
Update and insert successful!


In [2]:
from functionBigQuery import merge_review_records

In [3]:
merge_review_records(df_review)

Project is:  acme-987654


BadRequest: 400 Syntax error: Unexpected "=" at [6:32]; reason: invalidQuery, location: query, message: Syntax error: Unexpected "=" at [6:32]

Location: US
Job ID: b93f48d3-754a-4639-be25-81e7f64b8d01
